In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [5]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [ ]:

from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [9]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [10]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='1cb16687-2861-474a-acc8-aac2408d2652'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "what is langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'29d6ec8c-7a86-4598-b026-9e731e28ccb6': 'EjQKMgERTTIPA+7I7EhYx993NAdi3xmgZC1yu8hnfmGHcpCPuej4oLb7DCJkmJZQ3Klif8uV'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fad2f-12c5-7db0-aee3-fad180a09433-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'what is langchain-mcp-adapters library'}, 'id': '29d6ec8c-7a86-4598-b026-9e731e28ccb6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens': 24, 'total_tokens': 223, 'input_token_details': {'cache_read

## Online MCP

In [31]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "python",
            "args": [
               "resources/2.1_mcp_server.py",
            ]
        }
    }
)

tools = await client.get_tools()
tools

[StructuredTool(name='search_web', description='Search the web for information', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_webArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7b74c45f18a0>),
 StructuredTool(name='get_joke', description='Fetch a live random dark joke from JokeAPI. \nALWAYS use this tool whenever the user asks for a joke or entertainment, \ndo not invent your own joke.', args_schema={'properties': {}, 'title': 'get_jokeArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7b74b81996c0>)]

In [32]:
agent = create_agent(
    model=model,
    tools=tools,
)   

In [ ]:
question = HumanMessage(content="Who is the current president of the United States?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='Who is the current president of the United States?', additional_kwargs={}, response_metadata={}, id='e697ef87-b8a2-48f4-ade7-73ec62d5abc7'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "current president of the United States"}'}, '__gemini_function_call_thought_signatures__': {'feba102f-fc81-4640-8c1a-e0bb2720f284': 'EjQKMgERTTIPaIFCZpRIFilcjdafbHPR0P6gsGWP5BZQM7aPNJ2exPm9MrWZ0Jq1Sb93e5CZ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fad4f-f7c3-7112-b704-598efe649918-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'current president of the United States'}, 'id': 'feba102f-fc81-4640-8c1a-e0bb2720f284', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 21, 'total_tokens': 74, 'input_token_details': {'cache_read

In [37]:

question = HumanMessage(content="I am bored. Can you tell me a joke?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response['messages'][-1].content)

[{'extras': {'signature': 'EjQKMgERTTIPGdvPuaNjvZ76Xxoac0bqLd75auDC7caoqBBDF8zPePmJ1bw0VnjnpfPUIqOK'},
  'text': 'My ex had an accident. I told the paramedics the wrong blood type '
          "for her. She'll finally experience what rejection is really like.",
  'type': 'text'}]
